# Notebook 06 — Modelos Tuneados (Experimentos 3T y 4T)

## Objetivo

Entrenar XGBoost y MLP con **hiperparámetros óptimos** encontrados mediante walk-forward expanding window + Bayesian search (Optuna TPE). Comparar ambos modelos tuneados entre sí y contra los baselines del notebook 05.

| Experimento | Modelo         | Features                          | Tuning                        |
|-------------|----------------|-----------------------------------|-------------------------------|
| 3T          | XGBoost        | Técnicas + Macro + Sentiment (14) | Walk-forward Bayesian (Optuna) |
| 4T          | MLP            | Técnicas + Macro + Sentiment (14) | Walk-forward Bayesian (Optuna) |

**Input**: `data/processed/feature_matrix.csv` (construido en notebook 05).

**Split**: idéntico al resto del proyecto — Train 2008–2023 / Val 2024 / Test 2025.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

from src import models, utils, hyperparameters

utils.set_plot_style()

PROCESSED_DIR = "../data/processed"
MODELS_DIR    = "../models"

FEATURE_COLS = [
    "RSI_14", "MACD", "BB_position", "return_1d", "return_5d", "volume_change",
    "vix", "t10y2y", "fedfunds", "cpi", "unrate",
    "sentiment_mean", "sentiment_std", "news_count"
]

c:\Users\pepob\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carga de datos

Usamos la `feature_matrix.csv` generada en el notebook 05, que ya incluye las 14 features (técnicas + macro + sentiment) y el target.

In [2]:
df = pd.read_csv(f"{PROCESSED_DIR}/feature_matrix.csv", index_col=0, parse_dates=True)
df = df.dropna(subset=FEATURE_COLS)
print(f"Dataset: {df.shape} | {df.index[0].date()} → {df.index[-1].date()}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/feature_matrix.csv'

## Split temporal y escalado

Mismo split que en el resto del proyecto. El scaler se fitea solo sobre el train set.

| Conjunto | Período              |
|----------|----------------------|
| Train    | 2008–2023 (16 años)  |
| Val      | 2024 (1 año)         |
| Test     | 2025 (1 año)         |

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = models.temporal_split(
    df, features=FEATURE_COLS, target="target"
)
X_train_s, X_val_s, X_test_s, scaler = models.scale_features(X_train, X_val, X_test)

## Experimento 3T: XGBoost — Walk-Forward Bayesian Search

Buscamos los mejores hiperparámetros para XGBoost usando walk-forward expanding window sobre los años 2020–2024, optimizando AUC-ROC promedio.

| Hiperparámetro    | Rango de búsqueda       |
|-------------------|-------------------------|
| `max_depth`       | [3, 6]                  |
| `learning_rate`   | [0.01, 0.20] log        |
| `subsample`       | [0.6, 1.0]              |
| `colsample_bytree`| [0.6, 1.0]              |
| `min_child_weight`| [1, 7]                  |
| `reg_alpha`       | [0.0, 2.0]              |
| `reg_lambda`      | [0.5, 3.0]              |
| `gamma`           | [0.0, 2.0]              |

In [ ]:
best_params_xgb, study_xgb = hyperparameters.walk_forward_bayesian_search(
    df,
    features=FEATURE_COLS,
    target="target",
    n_trials=50,
)

In [ ]:
xgb_tuned = hyperparameters.train_xgboost_tuned(
    X_train_s, y_train, X_val_s, y_val, best_params_xgb
)

metricas_xgb = {}
for nombre, X, y in [("train", X_train_s, y_train), ("val", X_val_s, y_val), ("test", X_test_s, y_test)]:
    metricas_xgb[nombre] = models.evaluate_model(xgb_tuned, X, y, nombre)

models.save_model(xgb_tuned, scaler, "xgboost_tuned", MODELS_DIR)

## Experimento 4T: MLP — Walk-Forward Bayesian Search

**Pendiente**: implementar `walk_forward_bayesian_search_mlp` en `src/hyperparameters.py`.

El espacio de búsqueda sugerido ya está documentado en el docstring de esa función:
- `hidden_layer_sizes`: [(64,), (128,64), (256,128), (256,128,64)]
- `learning_rate_init`: [1e-4, 1e-2] log
- `alpha` (L2): [1e-5, 1e-1] log
- `activation`: ['relu', 'tanh']

Una vez implementada, descomentar la celda de búsqueda y correr el notebook desde acá.

In [ ]:
# COMPLETAR: implementar walk_forward_bayesian_search_mlp en src/hyperparameters.py
# y descomentar cuando esté listo:
#
# best_params_mlp, study_mlp = hyperparameters.walk_forward_bayesian_search_mlp(
#     df,
#     features=FEATURE_COLS,
#     target="target",
#     n_trials=50,
# )

best_params_mlp = None  # reemplazar con el dict de best_params cuando esté listo

In [ ]:
if best_params_mlp is not None:
    mlp_tuned = hyperparameters.train_mlp_tuned(X_train_s, y_train, best_params_mlp)

    metricas_mlp = {}
    for nombre, X, y in [("train", X_train_s, y_train), ("val", X_val_s, y_val), ("test", X_test_s, y_test)]:
        metricas_mlp[nombre] = models.evaluate_model(mlp_tuned, X, y, nombre)

    models.save_model(mlp_tuned, scaler, "mlp_tuned", MODELS_DIR)
else:
    print("[06] MLP pendiente de tuning — completar walk_forward_bayesian_search_mlp()")
    metricas_mlp = None
    mlp_tuned    = None

## Comparación: XGBoost Tuned vs MLP Tuned

Comparamos los dos modelos tuneados. Métricas principales: AUC-ROC (métrica de ranking, insensible al desbalance) y F1-score.

In [ ]:
filas = [
    {"Modelo": "XGBoost Tuned", "Split": s, **{k: v for k, v in m.items() if k != "split"}}
    for s, m in metricas_xgb.items()
]

if metricas_mlp is not None:
    filas += [
        {"Modelo": "MLP Tuned", "Split": s, **{k: v for k, v in m.items() if k != "split"}}
        for s, m in metricas_mlp.items()
    ]

tabla = pd.DataFrame(filas)
print("Comparación de modelos tuneados:")
display(tabla.set_index(["Modelo", "Split"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

modelos_roc = [("XGBoost Tuned", xgb_tuned, X_test_s, y_test)]
if mlp_tuned is not None:
    modelos_roc.append(("MLP Tuned", mlp_tuned, X_test_s, y_test))

for label, modelo, X, y in modelos_roc:
    y_score = modelo.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, y_score)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{label} (AUC={roc_auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", label="Aleatorio (AUC=0.500)")
ax.set_xlabel("Tasa de Falsos Positivos")
ax.set_ylabel("Tasa de Verdaderos Positivos")
ax.set_title("Curvas ROC — Modelos Tuneados (Test Set 2025)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()